# BÀI TẬP LỚN 02: PHÂN LOẠI NGUY CƠ TIỂU ĐƯỜNG BẰNG MẠNG NƠ-RON 5 TẦNG MLP
**Học phần:** Phát triển hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu bài thực hành:**
1. Tiền xử lý dữ liệu Pima Indians Diabetes, mở rộng 76 chiều đặc trưng (Polynomial Features & Inter-feature projection).
2. Tự cài đặt lớp DenseLayerNumPy và ReLUNumPy từ đầu đối chiếu với PyTorch.
3. Huấn luyện mạng truyền thẳng đa tầng 5 lớp (76 -> 128 -> 64 -> 32 -> 16 -> 1).
4. Đánh giá toàn diện các chỉ số: Accuracy, Precision, Recall, F1-Score, ROC-AUC và Ma trận nhầm lẫn.


In [ ]:
# Cell [1]: Nạp dữ liệu và Tiền xử lý mở rộng 76 chiều đặc trưng
import os, sys, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

candidate_paths = [
    '../data/diabetes.csv',
    'data/diabetes.csv',
    '../../data/diabetes.csv',
    '../New folder/diabetes.csv'
]
data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

print(f"[INFO] Nap du lieu tu: {data_path}")
df = pd.read_csv(data_path)
print(f"Kich thuoc du lieu goc: {df.shape}")

X_raw = df.drop('Outcome', axis=1).values
y_raw = df['Outcome'].values

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_raw)
np.random.seed(42)
extra_weights = np.random.randn(X_raw.shape[1], 76 - X_poly.shape[1])
X_extra = X_raw @ extra_weights
X_diab = np.hstack([X_poly, X_extra])

scaler = StandardScaler()
X_norm = scaler.fit_transform(X_diab)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y_raw, test_size=0.2, random_state=42, stratify=y_raw)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


In [ ]:
# Cell [2]: Cai dat tang Dense va ReLU tu dau bang NumPy (From Scratch)
class DenseLayerNumPy:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / in_features)
        self.b = np.zeros((1, out_features))
    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b
    def backward(self, dout, lr=0.01):
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0, keepdims=True)
        dx = np.dot(dout, self.W.T)
        self.W -= lr * self.dW
        self.b -= lr * self.db
        return dx

class ReLUNumPy:
    def forward(self, x):
        self.mask = (x > 0)
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask

print("[OK] Da hoan thanh cai dat NumPy Dense & ReLU tu dau.")


In [ ]:
# Cell [3]: Xay dung mo hinh 5 tang MLP tren PyTorch
class MLP5LayerDiabetes(nn.Module):
    def __init__(self, in_dim=76):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),     nn.ReLU(),
            nn.Linear(64, 32),      nn.ReLU(),
            nn.Linear(32, 16),      nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP5LayerDiabetes(in_dim=76)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tong so tham so cua mo hinh 5 tang: {total_params}")


In [ ]:
# Cell [4]: Huan luyen mo hinh voi BCEWithLogitsLoss va Adam Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

X_tr_t = torch.FloatTensor(X_train)
y_tr_t = torch.FloatTensor(y_train).unsqueeze(1)
X_te_t = torch.FloatTensor(X_test)

train_losses = []
for epoch in range(1, 21):
    model.train()
    optimizer.zero_grad()
    logits = model(X_tr_t)
    loss = criterion(logits, y_tr_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/20] - Train Loss: {loss.item():.4f}")

print("[OK] Huan luyen hoan tat!")


In [ ]:
# Cell [5]: Danh gia hieu nang va Truc quan hoa Ma tran nham lan
model.eval()
with torch.no_grad():
    test_logits = model(X_te_t)
    probs = torch.sigmoid(test_logits).numpy().flatten()
    preds = (probs >= 0.5).astype(int)

acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds)
rec = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
roc = roc_auc_score(y_test, probs)

print("=== KET QUA DANH GIA MO HINH 5 TANG MLP TREN TAP TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc:.4f}")

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Diabetes Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()
